# Module 1: Introduction to Big Data

**BigData Analytics Course — AKTN Magister Program**

---

## Learning Objectives
By the end of this module you will be able to:
1. Define Big Data and describe the **5 V's** (Volume, Velocity, Variety, Veracity, Value).
2. Describe the **Hadoop ecosystem** and its core components.
3. Explain the **CAP theorem** and its implications for distributed systems.
4. Identify real-world Big Data use-cases across industries.
5. Set up a **Google Colab** environment for subsequent modules.

---

**Estimated time:** 90 minutes  
**Prerequisites:** Basic Python knowledge

## 1. What Is Big Data?

Big Data refers to datasets that are **too large, fast, or complex** for traditional data-processing software to handle adequately. The concept is best captured by the **5 V's** framework:

| V | Definition | Example |
|---|------------|---------|
| **Volume** | Petabytes–Exabytes of data | Social media posts, sensor logs |
| **Velocity** | Data generated and processed at high speed | Stock tick data, IoT streams |
| **Variety** | Structured, semi-structured, and unstructured | CSV, JSON, video, audio |
| **Veracity** | Uncertainty and trustworthiness of data | Noisy sensor readings, fake reviews |
| **Value** | Business insight extracted from data | Customer segmentation, fraud detection |

> 💡 **Key insight:** The real goal of Big Data is extracting **Value** — the other four V's describe the challenges you must overcome.

In [ ]:
# ── Setup: install / import required packages ──────────────────────────────
# (These are pre-installed on Google Colab; run if using a local environment)
# !pip install matplotlib pandas numpy --quiet

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

print("✅ Libraries imported successfully.")

## 2. Visualising the 5 V's

In [ ]:
# Radar chart for the 5 V's
labels = ['Volume', 'Velocity', 'Variety', 'Veracity', 'Value']
num_vars = len(labels)

# Hypothetical scores (0-10) for three industry domains
scores = {
    'Finance':      [8, 9, 6, 7, 9],
    'Healthcare':   [7, 5, 9, 8, 8],
    'Social Media': [10, 10, 9, 5, 7],
}

angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

colors = ['#2196F3', '#4CAF50', '#FF9800']
for (domain, vals), color in zip(scores.items(), colors):
    vals = vals + vals[:1]
    ax.plot(angles, vals, color=color, linewidth=2, label=domain)
    ax.fill(angles, vals, color=color, alpha=0.15)

ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=13)
ax.set_ylim(0, 10)
ax.set_title('Big Data 5 V\'s by Industry Domain', size=15, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

## 3. The Hadoop Ecosystem

Apache Hadoop is the foundational open-source framework for distributed storage and processing of large datasets.

### Core Components

```
┌─────────────────────────────────────────────────────┐
│               Hadoop Ecosystem                      │
│  ┌──────────┐  ┌─────────┐  ┌──────────────────┐   │
│  │  HDFS    │  │  YARN   │  │   MapReduce      │   │
│  │(Storage) │  │(Resorc) │  │  (Processing)    │   │
│  └──────────┘  └─────────┘  └──────────────────┘   │
│  ┌──────────┐  ┌─────────┐  ┌──────────────────┐   │
│  │   Hive   │  │   Pig   │  │     HBase        │   │
│  │  (SQL)   │  │(Script) │  │   (NoSQL DB)     │   │
│  └──────────┘  └─────────┘  └──────────────────┘   │
│  ┌──────────┐  ┌─────────┐  ┌──────────────────┐   │
│  │  Spark   │  │  Kafka  │  │    ZooKeeper     │   │
│  │(In-mem.) │  │(Stream) │  │  (Coordination)  │   │
│  └──────────┘  └─────────┘  └──────────────────┘   │
└─────────────────────────────────────────────────────┘
```

| Component | Purpose |
|-----------|---------|
| **HDFS** | Hadoop Distributed File System — fault-tolerant distributed storage |
| **YARN** | Yet Another Resource Negotiator — resource management and job scheduling |
| **MapReduce** | Programming model for parallel processing |
| **Apache Hive** | SQL-like query language over HDFS |
| **Apache Spark** | In-memory distributed processing (100× faster than MapReduce) |
| **Apache Kafka** | Distributed event-streaming platform |
| **HBase** | Column-oriented NoSQL database on top of HDFS |

In [ ]:
# Simulated MapReduce Word Count — illustrating the MapReduce paradigm in Python
from collections import Counter
import re

sample_text = """
Big data is a term that describes large complex data sets.
Big data challenges include capturing data, data storage, data analysis,
search, sharing, transfer, visualization, querying, updating, information
privacy and data source. Big data was originally associated with three key
concepts: volume, variety, and velocity. When we handle big data we need
distributed computing frameworks like Apache Hadoop and Apache Spark.
"""

# MAP phase — tokenize and emit (word, 1) pairs
def map_phase(text):
    words = re.findall(r'\b[a-z]+\b', text.lower())
    return [(word, 1) for word in words]

# SHUFFLE phase — group by key
def shuffle_phase(pairs):
    grouped = {}
    for word, count in pairs:
        grouped.setdefault(word, []).append(count)
    return grouped

# REDUCE phase — sum counts per word
def reduce_phase(grouped):
    return {word: sum(counts) for word, counts in grouped.items()}

mapped   = map_phase(sample_text)
shuffled = shuffle_phase(mapped)
result   = reduce_phase(shuffled)

# Display top-15 words
top15 = sorted(result.items(), key=lambda x: x[1], reverse=True)[:15]
words_plot, counts_plot = zip(*top15)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(words_plot, counts_plot, color='steelblue', edgecolor='white')
ax.set_xlabel('Word', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('MapReduce Word Count — Top 15 Words', fontsize=14)
for bar, val in zip(bars, counts_plot):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            str(val), ha='center', fontsize=10)
plt.tight_layout()
plt.show()
print("MapReduce simulation complete!")

## 4. CAP Theorem

The **CAP Theorem** (Brewer, 2000) states that a distributed data store can guarantee **at most two** of the following three properties simultaneously:

| Property | Meaning |
|----------|---------|
| **C**onsistency | Every read receives the most recent write |
| **A**vailability | Every request receives a response (not necessarily latest data) |
| **P**artition Tolerance | System continues operating despite network partitions |

```
          Consistency
               △
              /|\
             / | \
    CA      /  |  \   CP
           /   |   \
          /    |    \
Availability──────────Partition
              AP     Tolerance
```

**Practical implications:**
- **CP systems** (e.g., HBase, Zookeeper): consistent but may be unavailable during partitions.
- **AP systems** (e.g., Cassandra, CouchDB): always available but may serve stale data.
- **CA systems** (e.g., traditional RDBMS): consistent and available but cannot tolerate partitions — only feasible on single nodes.

In [ ]:
# Visual: CAP Theorem Venn Diagram
from matplotlib.patches import Circle, FancyArrowPatch

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

circle_params = dict(radius=2.8, fill=True, alpha=0.25, linewidth=2)
c1 = Circle((3.5, 6.5), color='#2196F3', **circle_params)   # Consistency
c2 = Circle((6.5, 6.5), color='#4CAF50', **circle_params)   # Partition Tolerance
c3 = Circle((5.0, 4.0), color='#FF9800', **circle_params)   # Availability

for c in [c1, c2, c3]:
    ax.add_patch(c)

ax.text(2.0, 8.2,  'Consistency',          fontsize=13, ha='center', color='#1565C0', fontweight='bold')
ax.text(8.0, 8.2,  'Partition\nTolerance', fontsize=13, ha='center', color='#2E7D32', fontweight='bold')
ax.text(5.0, 2.0,  'Availability',         fontsize=13, ha='center', color='#E65100', fontweight='bold')

ax.text(2.8, 5.6,  'CA\n(RDBMS)',   fontsize=10, ha='center', style='italic')
ax.text(7.2, 5.6,  'CP\n(HBase)',   fontsize=10, ha='center', style='italic')
ax.text(5.0, 5.8,  'CP/AP',        fontsize=10, ha='center', style='italic')
ax.text(5.0, 4.6,  'AP\n(Cassandra)', fontsize=10, ha='center', style='italic')

ax.set_title('CAP Theorem — Choose Two', fontsize=16, pad=10)
plt.tight_layout()
plt.show()

## 5. Big Data in Industry — Use Cases

| Industry | Big Data Application | Business Value |
|----------|---------------------|----------------|
| Finance | Real-time fraud detection | Prevent billions in losses |
| Healthcare | Genomic data analysis, patient risk scoring | Earlier diagnosis |
| Retail / E-commerce | Recommendation engines | +25–35% revenue lift |
| Manufacturing | Predictive maintenance (IoT) | Reduced downtime |
| Transportation | Route optimisation (Uber/Grab) | Lower fuel costs |
| Telecommunications | Network anomaly detection | Better QoS, churn reduction |
| Government | Smart city traffic management | Safer, greener cities |

In [ ]:
# Data growth timeline — how much data the world generates
years      = [2010, 2012, 2014, 2016, 2018, 2020, 2022, 2024, 2026]
data_zb    = [2,    4,    6,    16,   33,   64,   97,   120,  175]  # Zettabytes (approximate)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(years, data_zb, alpha=0.3, color='#2196F3')
ax.plot(years, data_zb, 'o-', color='#1565C0', linewidth=2.5, markersize=7)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Data Generated (Zettabytes)', fontsize=12)
ax.set_title('Global Data Sphere Growth (IDC Estimates)', fontsize=14)
ax.set_xticks(years)
for x, y in zip(years, data_zb):
    ax.annotate(f'{y} ZB', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 6. Google Colab Environment Setup

Google Colab provides free access to:
- Python 3 runtime
- NVIDIA GPUs / TPUs
- Pre-installed data science libraries
- Up to 15 GB RAM (standard) or 52 GB (Colab Pro)

### Useful Colab Tips
```python
# Check GPU availability
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install a package at runtime
!pip install pyspark --quiet
```

In [ ]:
# Environment diagnostic — verify what's available in our runtime
import sys
import platform

print("=" * 50)
print("ENVIRONMENT DIAGNOSTIC")
print("=" * 50)
print(f"Python version : {sys.version.split()[0]}")
print(f"Platform       : {platform.system()} {platform.release()}")

libs = [
    ("numpy",       "numpy"),
    ("pandas",      "pandas"),
    ("matplotlib",  "matplotlib"),
    ("sklearn",     "scikit-learn"),
]

print("\nLibrary versions:")
for mod, name in libs:
    try:
        m = __import__(mod)
        ver = getattr(m, '__version__', 'unknown')
        print(f"  {name:<20} {ver}")
    except ImportError:
        print(f"  {name:<20} NOT INSTALLED")

print("\n✅ Environment check complete.")

## 7. Summary & Key Takeaways

| Concept | Summary |
|---------|---------|
| **5 V's** | Volume, Velocity, Variety, Veracity, Value |
| **Hadoop** | Open-source framework: HDFS (storage) + YARN (resources) + MapReduce/Spark (processing) |
| **CAP Theorem** | Choose two of Consistency, Availability, Partition Tolerance |
| **Use Cases** | Finance, Healthcare, Retail, Manufacturing, Government |
| **Google Colab** | Free cloud Python environment — ideal for Big Data prototyping |

## 📝 Exercises

1. **Conceptual:** For each industry in the use-case table, identify which two CAP properties are most important and justify your choice.
2. **Coding:** Extend the MapReduce Word Count to ignore common stop-words (`the`, `a`, `is`, `and`, …) and display a word cloud.
3. **Research:** Find a real-world dataset (e.g., from [Kaggle](https://www.kaggle.com/datasets)) that demonstrates at least 3 of the 5 V's. Describe it in 200 words.

---
➡️ **Next:** [Module 2 — Data Processing with Python](Module_02_Data_Processing_with_Python.ipynb)